## ForgettingFairly — Pilot
**Hypothesis:** Continual learning forgets Task 1 unevenly across demographic subgroups.

| Setting | Value |
|---|---|
| Dataset | HAM10000 (`kmader/skin-cancer-mnist-ham10000`) |
| Task 1 | `vidir_modern` (ViDIR Vienna) |
| Task 2 | `rosendahl` (QIMR Queensland) |
| Sensitive attr | `sex` (male / female) |
| Backbone | ResNet50 on RadImageNet — **frozen** |
| Expected runtime | ~20 min on T4 |

> ⚠️ Before running: replace `YOUR_USERNAME` in Cell 1 with your GitHub username.

### Cell 1 — Clone repo

In [ ]:
import subprocess, os

REPO_URL = "https://github.com/YOUR_USERNAME/forgetting-fairly.git"  # ← update this

result = subprocess.run(
    ["git", "clone", REPO_URL, "/kaggle/working/forgetting-fairly"],
    capture_output=True, text=True
)
print(result.stdout or "(no stdout)")
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("git clone failed – did you update REPO_URL?")

os.chdir("/kaggle/working/forgetting-fairly")
print("Working dir:", os.getcwd())
!pip install scikit-learn --upgrade -q
print("\n✅ ready")

### Cell 2 — Verify all inputs exist

In [ ]:
import os

checks = {
    "Metadata CSV":    "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv",
    "Images part 1":   "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1",
    "Images part 2":   "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2",
    "RadImageNet wts": "/kaggle/input/models/albertoeusebio/resnet_radimagenet/pytorch/default/1/ResNet50.pt",
}

all_ok = True
for name, path in checks.items():
    exists = os.path.exists(path)
    status = "✅" if exists else "❌ MISSING"
    if os.path.isdir(path):
        n = len(os.listdir(path))
        print(f"{status}  {name}: {path}  ({n} files)")
    else:
        size = f"{os.path.getsize(path)/1e6:.1f} MB" if exists else ""
        print(f"{status}  {name}: {path}  {size}")
    if not exists:
        all_ok = False

if all_ok:
    print("\n✅ All inputs present – good to go")
else:
    print("\n❌ Some inputs are missing – check dataset/model attachments")

### Cell 3 — Sanity check metadata
Verify the source column values that define our two tasks.

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv")
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

# This dataset uses 'dataset' as the source column (not 'datasource')
src_col = "dataset" if "dataset" in df.columns else "datasource"
print(f"\nSource column = '{src_col}'")
print(df[src_col].value_counts())

print("\nClass distribution:")
print(df["dx"].value_counts())

print("\nSex distribution:")
print(df["sex"].value_counts(dropna=False))

# Confirm our two task sources exist in mel/nv subset
mel_nv = df[df["dx"].isin(["mel", "nv"])]
t1 = mel_nv[mel_nv[src_col] == "vidir_modern"]
t2 = mel_nv[mel_nv[src_col] == "rosendahl"]
print(f"\nTask 1 (vidir_modern):  {len(t1)} mel+nv samples")
print(f"Task 2 (rosendahl):     {len(t2)} mel+nv samples")

if len(t1) == 0 or len(t2) == 0:
    raise ValueError("One task source is empty – check source column values above")

### Cell 4 — Symlink images into a flat directory
Symlinks cost zero disk space and zero copy time.

In [ ]:
import os

IMG_DIR = "./data/HAM10000_images"
os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs("./data", exist_ok=True)

# Symlink metadata
META_DST = "./data/HAM10000_metadata.csv"
if not os.path.exists(META_DST):
    os.symlink("/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv", META_DST)
print(f"Metadata → {META_DST}")

# Symlink images from both parts
n_new = 0
for part_path in ["/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1", "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2"]:
    if not os.path.isdir(part_path):
        print(f"⚠️  {part_path} not found")
        continue
    for fname in os.listdir(part_path):
        dst = os.path.join(IMG_DIR, fname)
        if not os.path.exists(dst):
            os.symlink(os.path.join(part_path, fname), dst)
            n_new += 1

total = len(os.listdir(IMG_DIR))
print(f"{n_new} new symlinks  |  {total} total images in {IMG_DIR}")
print("✅ image directory ready")

### Cell 5 — GPU check

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print("✅ GPU ready")
else:
    print("⚠️  No GPU – enable it in notebook settings (≥10× slower on CPU)")

### Cell 6 — Run the pilot (~20 min on T4)

In [ ]:
!python pilot.py \
    --epochs 10 \
    --batch_size 64 \
    --lr 1e-3 \
    --img_dir ./data/HAM10000_images \
    --metadata ./data/HAM10000_metadata.csv \
    --weights "/kaggle/input/models/albertoeusebio/resnet_radimagenet/pytorch/default/1/ResNet50.pt"

### Cell 7 — Results

In [ ]:
import json

with open("./results/pilot_results.json") as f:
    r = json.load(f)

m1 = r["metrics_after_task1"]
m2 = r["metrics_after_task2"]

print("=" * 58)
print("  PILOT RESULTS")
print("=" * 58)
print(f"  {'Metric':<32} {'After T1':>8}  {'After T2':>8}  {'Δ':>7}")
print(f"  {'-'*52}")

def row(label, v1, v2):
    d  = (v2-v1) if (v1 is not None and v2 is not None) else None
    v1s = f"{v1:.4f}" if v1 is not None else "   N/A"
    v2s = f"{v2:.4f}" if v2 is not None else "   N/A"
    ds  = f"{d:+.4f}" if d  is not None else "   N/A"
    print(f"  {label:<32} {v1s:>8}  {v2s:>8}  {ds:>7}")

row("Overall AUC",           m1["overall"]["auc"],  m2["overall"]["auc"])
row("Overall acc",           m1["overall"]["acc"],  m2["overall"]["acc"])
row("Male   AUC",            m1["male"]["auc"],     m2["male"]["auc"])
row("Female AUC",            m1["female"]["auc"],   m2["female"]["auc"])
row("Male   acc",            m1["male"]["acc"],     m2["male"]["acc"])
row("Female acc",            m1["female"]["acc"],   m2["female"]["acc"])
row("Male   TPR",            m1["male"]["tpr"],     m2["male"]["tpr"])
row("Female TPR",            m1["female"]["tpr"],   m2["female"]["tpr"])
print(f"  {'-'*52}")
row("EOD gap |TPR_m-TPR_f|", m1["eod_gap"],         m2["eod_gap"])
row("Acc gap |acc_m-acc_f|", m1["acc_gap"],          m2["acc_gap"])

delta_eod   = m2["eod_gap"] - m1["eod_gap"]
diff_forget = abs(
    (m2["male"]["acc"] - m1["male"]["acc"]) -
    (m2["female"]["acc"] - m1["female"]["acc"])
)

print(f"\n  Δ EOD gap:               {delta_eod:+.4f}  {'✅' if delta_eod > 0 else '⚠️'}")
print(f"  Differential forgetting: {diff_forget:.4f}  {'✅' if diff_forget > 0.02 else '⚠️'}")

if delta_eod > 0 and diff_forget > 0.02:
    print("\n  🚀 HYPOTHESIS CONFIRMED")
elif delta_eod > 0 or diff_forget > 0.02:
    print("\n  🔶 PARTIAL SIGNAL")
else:
    print("\n  🔴 WEAK SIGNAL – check task splits")

### Cell 8 — Save outputs
Files in `/kaggle/working/` appear in the Output tab and can be downloaded.

In [ ]:
import shutil, os, json

os.makedirs("/kaggle/working/outputs", exist_ok=True)

shutil.copy("./results/pilot_results.json",
            "/kaggle/working/outputs/pilot_results.json")
shutil.copy("./checkpoints/after_task1.pt",
            "/kaggle/working/outputs/after_task1.pt")

with open("./results/pilot_results.json") as f:
    r = json.load(f)

summary = {
    "delta_eod_gap":    r["metrics_after_task2"]["eod_gap"] - r["metrics_after_task1"]["eod_gap"],
    "delta_acc_male":   r["metrics_after_task2"]["male"]["acc"] - r["metrics_after_task1"]["male"]["acc"],
    "delta_acc_female": r["metrics_after_task2"]["female"]["acc"] - r["metrics_after_task1"]["female"]["acc"],
    "auc_t1": r["metrics_after_task1"]["overall"]["auc"],
    "auc_t2": r["metrics_after_task2"]["overall"]["auc"],
    "config": r["config"],
}
with open("/kaggle/working/outputs/summary.json","w") as f:
    json.dump(summary, f, indent=2)

for fname in sorted(os.listdir("/kaggle/working/outputs")):
    sz = os.path.getsize(f"/kaggle/working/outputs/{fname}")
    print(f"  {fname}  ({sz/1e6:.1f} MB)")
print("\n✅ Download from the Output tab →")